# U-JEPA Phase 2: LLM-JEPA + SIGReg on Spider

Two-arm comparison on a frozen NF4 Qwen3-14B:
  arm A = LoRA + CE only (baseline)
  arm B = LoRA + CE + LLM-JEPA cosine + SIGReg (treatment)

Gates: (1) treatment Spider EM minus baseline EM >= +0.02, (2) hidden-state covariance condition number on the treatment arm < 100.

Output: `/kaggle/working/results/phase2_jepa_aux.json`.

Runtime: single T4, Internet On. The base model lives on cuda:0 via device_map. Qwen3-14B is ungated so HF_TOKEN is optional.

In [ ]:
# Move the HF cache off the 20 GB /kaggle/working quota; Qwen3-14B alone is ~28 GB.
import shutil, os, subprocess
stale = '/kaggle/working/hf_cache'
if os.path.isdir(stale):
    print(f'removing stale cache at {stale}')
    shutil.rmtree(stale)
os.makedirs('/tmp/hf_cache', exist_ok=True)
try:
    print(subprocess.check_output(['df', '-h', '/kaggle/working', '/tmp']).decode())
except Exception as e:
    print(f'df check skipped: {e}')

In [ ]:
import subprocess, os, sys
if not os.path.exists('/kaggle/working/U-JEPA'):
    subprocess.run(['git', 'clone', 'https://github.com/kartikshirode/U-JEPA.git',
                    '/kaggle/working/U-JEPA'], check=True)
else:
    subprocess.run(['git', '-C', '/kaggle/working/U-JEPA', 'pull'], check=True)
os.chdir('/kaggle/working/U-JEPA')
# Install without -q so any pip failure shows up in the kernel log.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r',
                'requirements-kaggle.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.'], check=True)

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF_TOKEN loaded from Kaggle secrets')
except Exception as e:
    print(f'No HF_TOKEN secret ({e}). Qwen3-14B is ungated so this is fine.')
os.environ['HF_HOME'] = '/tmp/hf_cache'
os.environ['HF_HUB_CACHE'] = '/tmp/hf_cache'
os.environ['TRANSFORMERS_CACHE'] = '/tmp/hf_cache'
print(f'HF_HOME={os.environ["HF_HOME"]}')

In [ ]:
import torch
print(f'torch {torch.__version__}, cuda {torch.version.cuda}, GPUs: {torch.cuda.device_count()}')
assert torch.cuda.device_count() >= 1, 'No CUDA device; enable GPU accelerator.'
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'GPU {i}: {p.name}, {p.total_memory // (1024**2)} MiB')

In [ ]:
import subprocess, sys, os
env = os.environ.copy()
env['HF_HOME'] = '/tmp/hf_cache'
env['HF_HUB_CACHE'] = '/tmp/hf_cache'
env['TRANSFORMERS_CACHE'] = '/tmp/hf_cache'
subprocess.run([sys.executable, 'scripts/03_train_jepa_aux_phase2.py'], check=True, env=env)

In [ ]:
import json
from pathlib import Path
p = Path('/kaggle/working/results/phase2_jepa_aux.json')
if p.exists():
    print(json.dumps(json.loads(p.read_text()), indent=2))
else:
    print('No results file yet')